In [3]:
# Simple test script
try:
    # Import the module
    from running_phase6pipeline import TradingBot
    
    # Create bot instance
    bot = TradingBot(enable_live_trading=False)
    print("✓ TradingBot created successfully")
    
    # Get status
    status = bot.get_status()
    print(f"Status: {status}")
    
    # Run a single cycle
    result = bot.run_single_cycle()
    print(f"Trade result: {result}")
    
    # Shutdown
    bot.shutdown()
    print("✓ All operations completed successfully")
    
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()


# debug_test.py - FIXED VERSION
import sys

# First, let's check if the file has the correct __init__ method
with open('running_phase6pipeline.py', 'r') as f:
    content = f.read()
    
# Check for TradingBot class definition
print("Searching for TradingBot class...")
if 'class TradingBot:' in content:
    print("✓ Found TradingBot class")
    
    # Find the __init__ method
    import re
    init_pattern = r'def\s+__init__\s*\(self'
    matches = re.findall(init_pattern, content)
    
    print(f"Found __init__ methods: {matches}")
    
    # Specifically check TradingBot's __init__
    trading_bot_section = re.search(r'class TradingBot:.*?(?=class|\Z)', content, re.DOTALL)
    if trading_bot_section:
        tb_content = trading_bot_section.group(0)
        tb_init = re.search(r'def\s+(__init__)\s*\(self', tb_content)  # ← FIXED REGEX
        if tb_init:
            print(f"TradingBot has '{tb_init.group(1)}' method")
        else:
            print("✗ TradingBot has NO __init__ method!")
else:
    print("✗ TradingBot class not found!")

# CLEANED — safer, idempotent data preprocessing
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Add project root to path
project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

# ==================== DATA PREPROCESSING ====================
print("=" * 60)
print("STARTING PHASE 6 TRADING PIPELINE INTEGRATION")
print("=" * 60)

try:
    # Load data
    raw_path = 'data/raw/master.csv'
    data = pd.read_csv(raw_path)
    print(f"✓ Loaded data from {raw_path}")
    print(f"  Shape: {data.shape}")
    
except FileNotFoundError:
    print("✗ ERROR: Could not find master.csv")
    print("Creating sample data for testing...")
    
    # Create sample data for testing if file doesn't exist
    np.random.seed(42)
    dates = pd.date_range(start='2023-01-01', end='2024-01-01', freq='H')
    data = pd.DataFrame({
        'Time (UTC)': dates,
        'Open': 140 + np.cumsum(np.random.randn(len(dates)) * 0.1),
        'High': 140 + np.cumsum(np.random.randn(len(dates)) * 0.1) + np.random.rand(len(dates)) * 0.2,
        'Low': 140 + np.cumsum(np.random.randn(len(dates)) * 0.1) - np.random.rand(len(dates)) * 0.2,
        'Close': 140 + np.cumsum(np.random.randn(len(dates)) * 0.1),
        'Volume': np.random.randint(1000, 10000, len(dates))
    })
    print("✓ Created sample data for testing")

# 1) Normalize column names (strip whitespace, lowercase optional)
data.columns = [c.strip() for c in data.columns]
print("\n=== COLUMNS FOUND AFTER STRIP ===")
print(data.columns.tolist())
print("=================================\n")

# 2) If Time (UTC) present — parse it and rename to Date (UTC)
if 'Time (UTC)' in data.columns:
    data['Date'] = pd.to_datetime(data['Time (UTC)'], utc=True)
    print("✓ Parsed 'Time (UTC)' column")
elif 'Date' in data.columns:
    data['Date'] = pd.to_datetime(data['Date'], utc=True)
    print("✓ Parsed 'Date' column")
else:
    print("Warning: no Time column found — creating dummy date")
    data['Date'] = pd.date_range(start='2023-01-01', periods=len(data), freq='H')

# --- Technical Indicators ---
print("\n--- Creating Technical Indicators ---")
data['MA10'] = data['Close'].rolling(10).mean()
data['MA20'] = data['Close'].rolling(20).mean()
data['MA50'] = data['Close'].rolling(50).mean()
data['EMA10'] = data['Close'].ewm(span=10, adjust=False).mean()
data['EMA20'] = data['Close'].ewm(span=20, adjust=False).mean()
data['Volatility'] = data['Close'].rolling(20).std()

delta = data['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss.replace(0, pd.NA)
data['RSI'] = 100 - (100 / (1 + rs))

data['Close_lag1'] = data['Close'].shift(1)
data['Close_lag2'] = data['Close'].shift(2)
data['Close_lag3'] = data['Close'].shift(3)

# Drop rows with NaNs introduced by indicators
initial_len = len(data)
data = data.dropna().reset_index(drop=True)
print(f"✓ Created indicators, dropped {initial_len - len(data)} NaN rows")

# Create signals
data['Signal'] = 0
data.loc[data['Close'] > data['Close_lag1'], 'Signal'] = 1
data.loc[data['Close'] < data['Close_lag1'], 'Signal'] = -1

# Feature columns
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'MA10', 'MA20', 'MA50', 'EMA10', 'EMA20',
    'Volatility', 'RSI', 'Close_lag1', 'Close_lag2', 'Close_lag3'
]

# Validate feature existence
missing = [c for c in feature_cols if c not in data.columns]
if missing:
    print(f"Warning: Missing feature columns: {missing}")
    # Keep only available features
    feature_cols = [c for c in feature_cols if c in data.columns]
    print(f"Using available features: {feature_cols}")

features = data[feature_cols]
target = data['Signal']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, shuffle=False)
print(f"\n✓ Data split complete:")
print(f"  Training rows: {len(X_train)}")
print(f"  Testing rows: {len(X_test)}")

# Simple rule-based evaluation for sanity
def generate_signals(df):
    signals = []
    for _, r in df.iterrows():
        if r['Close'] > r['Close_lag1']:
            signals.append(1)
        elif r['Close'] < r['Close_lag1']:
            signals.append(-1)
        else:
            signals.append(0)
    return signals

train_signals = generate_signals(X_train)
test_signals = generate_signals(X_test)

def evaluate(pred, actual):
    correct = sum(1 for p, a in zip(pred, actual) if p == a)
    return correct / len(actual) * 100

train_acc = evaluate(train_signals, y_train.tolist())
test_acc = evaluate(test_signals, y_test.tolist())
print(f"\n✓ Rule-based model performance:")
print(f"  Training accuracy: {train_acc:.2f}%")
print(f"  Testing accuracy:  {test_acc:.2f}%")

# Save cleaned file for later use
clean_path = 'data/raw/master_clean.csv'
data.to_csv(clean_path, index=False)
print(f"\n✓ Saved cleaned data to {clean_path}")

# ==================== IMPORT AND RUN PHASE 6 PIPELINE ====================
print("\n" + "=" * 60)
print("INTEGRATING WITH PHASE 6 TRADING PIPELINE")
print("=" * 60)

try:
    # Import from the running_phase6pipeline.py file
    from running_phase6pipeline import create_trading_bot, run_quick_test
    
    print("✓ Successfully imported Phase 6 pipeline functions")
    
    # 1. Create a bot instance (simulation mode by default)
    print("\n1. Creating trading bot (simulation mode)...")
    bot = create_trading_bot(enable_live_trading=False)
    print("   ✓ Bot created successfully")
    
    # 2. Get initial status
    print("\n2. Getting initial status...")
    status = bot.get_status()
    print(f"   Pipeline Status: {status['pipeline']}")
    print(f"   Trading Symbols: {status['config']['symbols']}")
    print(f"   Live Trading: {'ENABLED ⚠' if status['config']['live_trading'] else 'DISABLED (safe)'}")
    
    # 3. Run a single cycle
    print("\n3. Running single trading cycle...")
    symbol = "USDJPYm"  # Use the symbol from config
    result = bot.run_single_cycle(symbol)
    
    if result:
        print(f"   ✓ Trade executed for {symbol}:")
        print(f"     Signal: {'BUY' if result['signal'] == 1 else 'SELL'}")
        print(f"     Size: {result['position_size']:.2f} lots")
        print(f"     Entry Price: {result['entry_price']}")
        print(f"     Stop Loss: {result['stop_loss']}")
        print(f"     Take Profit: {result['take_profit']}")
    else:
        print(f"   ⓘ No trade executed for {symbol} (no signal or checks failed)")
    
    # 4. Get updated performance
    print("\n4. Getting performance summary...")
    status = bot.get_status()
    if 'performance' in status:
        print("   Performance Metrics:")
        for key, value in status['performance'].items():
            print(f"     {key}: {value}")
    
    # 5. Optional: Run quick test
    print("\n5. Running quick test...")
    run_quick_test()
    
    # 6. Optional: Run continuous mode (uncomment to use)
    # print("\n6. Starting continuous trading (press Ctrl+C to stop)...")
    # bot.run_continuous(interval=30, max_runs=5)
    
    # 7. Shutdown
    print("\n7. Shutting down bot...")
    bot.shutdown()
    print("   ✓ Bot shutdown complete")
    
    print("\n" + "=" * 60)
    print("PHASE 6 PIPELINE EXECUTION COMPLETE ✓")
    print("=" * 60)
    
except ImportError as e:
    print(f"\n✗ ERROR: Could not import from running_phase6pipeline.py")
    print(f"  Error: {e}")
    print(f"\n  Make sure running_phase6pipeline.py is in your project root.")
    print(f"  Current working directory: {os.getcwd()}")
    print(f"  Files in directory: {os.listdir('.')}")
    
except Exception as e:
    print(f"\n✗ ERROR during pipeline execution: {e}")
    import traceback
    traceback.print_exc()

# ==================== ALTERNATIVE: DIRECT PIPELINE ACCESS ====================
print("\n" + "=" * 60)
print("ALTERNATIVE DIRECT ACCESS TO PIPELINE")
print("=" * 60)

try:
    # Alternative: Import the Phase6_PipelineOrchestrator directly
    from running_phase6pipeline import Phase6_PipelineOrchestrator, Phase6Config
    
    # Create custom configuration
    class MyConfig(Phase6Config):
        SYMBOLS = ["USDJPYm"]
        ENABLE_LIVE_TRADING = False
        PIPELINE_RUN_INTERVAL = 30  # 30 seconds between cycles
        MAX_PIPELINE_RUNS = 3  # Only run 3 cycles for testing
    
    print("Creating custom pipeline...")
    pipeline = Phase6_PipelineOrchestrator(MyConfig)
    
    # Run a few cycles manually
    print("\nRunning 3 manual cycles...")
    for i in range(3):
        print(f"\n--- Cycle {i+1} ---")
        result = pipeline.run_pipeline_cycle()
        if result:
            print(f"  Trade: {result['symbol']} {'BUY' if result['signal'] == 1 else 'SELL'}")
        else:
            print("  No trade")
        
        # Update and show performance
        perf = pipeline.portion5.update_performance()
        if perf:
            print(f"  Equity: ${perf['equity']:.2f}, Return: {perf['return_pct']:.2f}%")
    
    # Get final status
    print("\nFinal pipeline status:")
    status = pipeline.get_pipeline_status()
    print(f"  Total positions: {status['portfolio']['total_positions'] if 'portfolio' in status else 0}")
    
    # Shutdown
    pipeline.shutdown()
    print("\n✓ Direct pipeline test complete")
    
except Exception as e:
    print(f"\n✗ Alternative test failed: {e}")

# ==================== SUMMARY ====================
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"1. Data preprocessing complete:")
print(f"   - Original data shape: {data.shape}")
print(f"   - Features created: {len(feature_cols)}")
print(f"   - Rule-based accuracy: {test_acc:.1f}%")
print(f"\n2. Phase 6 Pipeline:")
print(f"   - Bot initialized: ✓")
print(f"   - Single cycle executed: ✓")
print(f"   - Simulation mode: ✓ (no real trades)")
print(f"\n3. Next steps:")
print(f"   - Train ML model on the prepared features")
print(f"   - Save model to 'models/model_latest.joblib'")
print(f"   - Enable live trading in Phase6Config")
print(f"   - Run continuous mode for automated trading")

print("\n✓ Notebook execution complete!")

2025-12-05 10:24:26,535 | Phase6 | INFO | ============================================================
2025-12-05 10:24:26,537 | Phase6 | INFO | Initializing Advanced 6-Phase Trading Pipeline v1.0.0
2025-12-05 10:24:26,545 | Phase6 | INFO | ============================================================
2025-12-05 10:25:32,324 | Phase6 | WARNING | MT5 not available - running in simulation mode
2025-12-05 10:25:32,325 | Portion1 | INFO | Signal model loaded (placeholder)
2025-12-05 10:27:43,848 | Portion3 | WARNING | MT5 initialization failed - running in simulation mode
2025-12-05 10:27:43,849 | Phase6 | INFO | All 6 portions initialized successfully
2025-12-05 10:27:43,850 | Phase6 | INFO | Phase 6 Pipeline initialized successfully
2025-12-05 10:28:49,621 | Phase6 | INFO | Starting pipeline cycle for USDJPYm


✓ TradingBot created successfully
Status: {'timestamp': '2025-12-05T10:27:43.851745', 'pipeline': 'Running', 'config': {'symbols': ['USDJPYm'], 'live_trading': False, 'risk_per_trade': 0.02}, 'performance': {}, 'portfolio': {'total_positions': 0, 'total_symbols': 0, 'total_volume': 0, 'total_profit': 0, 'symbols': {}}, 'live_trading': False}


2025-12-05 10:29:55,390 | Portion1 | ERROR | MT5 not initialized
2025-12-05 10:29:55,391 | Phase6 | INFO | No trading signal for USDJPYm (confidence: 0.00)
2025-12-05 10:29:55,393 | Phase6 | INFO | Shutting down Phase 6 Pipeline...


Trade result: None


2025-12-05 10:31:00,959 | Phase6 | ERROR | Failed to save pipeline state: [Errno 2] No such file or directory: 'state/pipeline_state.pkl'
2025-12-05 10:31:00,960 | Phase6 | INFO | Phase 6 Pipeline shutdown complete


...

   ⓘ No trade executed for USDJPYm (no signal or checks failed)

4. Getting performance summary...
   Performance Metrics:

5. Running quick test...
PHASE 6 PIPELINE QUICK TEST

1. Running single cycle...
No trade executed (no signal or checks failed)

2. Getting status...
  Pipeline: Running
  Live Trading: DISABLED (safe)
  Symbols: ['USDJPYm']
  Performance:

3. Shutting down...
QUICK TEST COMPLETED SUCCESSFULLY

7. Shutting down bot...
   ✓ Bot shutdown complete

PHASE 6 PIPELINE EXECUTION COMPLETE ✓

ALTERNATIVE DIRECT ACCESS TO PIPELINE
Creating custom pipeline...


2025-12-05 10:37:03,369 | Phase6 | INFO | ============================================================
2025-12-05 10:37:03,370 | Phase6 | INFO | Initializing Advanced 6-Phase Trading Pipeline v1.0.0
2025-12-05 10:37:03,372 | Phase6 | INFO | ============================================================
2025-12-05 10:37:03,399 | Portion1 | INFO | Signal model loaded (placeholder)
2025-12-05 10:37:03,446 | Portion3 | INFO | MT5 initialized successfully
2025-12-05 10:37:03,448 | Phase6 | INFO | All 6 portions initialized successfully
2025-12-05 10:37:03,450 | Phase6 | INFO | Phase 6 Pipeline initialized successfully
2025-12-05 10:37:03,451 | Phase6 | INFO | Starting pipeline cycle for USDJPYm
2025-12-05 10:37:03,489 | Portion1 | INFO | Signal generated for USDJPYm: 0 (confidence: 0.50)
2025-12-05 10:37:03,491 | Phase6 | INFO | No trading signal for USDJPYm (confidence: 0.50)
2025-12-05 10:37:03,541 | Phase6 | INFO | Starting pipeline cycle for USDJPYm
2025-12-05 10:37:03,572 | Portion1 | IN


Running 3 manual cycles...

--- Cycle 1 ---
  No trade
  Equity: $9981.91, Return: -0.18%

--- Cycle 2 ---
  No trade
  Equity: $9981.91, Return: -0.18%

--- Cycle 3 ---
  No trade


2025-12-05 10:37:03,670 | Phase6 | INFO | Shutting down Phase 6 Pipeline...
2025-12-05 10:37:03,686 | Phase6 | INFO | MT5 connection closed
2025-12-05 10:37:03,688 | Phase6 | ERROR | Failed to save pipeline state: [Errno 2] No such file or directory: 'state/pipeline_state.pkl'
2025-12-05 10:37:03,694 | Portion5 | INFO | Performance report saved: logs/performance_20251205_103703.json
2025-12-05 10:37:03,696 | Phase6 | INFO | Phase 6 Pipeline shutdown complete


  Equity: $9981.91, Return: -0.18%

Final pipeline status:
  Total positions: 0

✓ Direct pipeline test complete

SUMMARY
1. Data preprocessing complete:
   - Original data shape: (450241, 18)
   - Features created: 15
   - Rule-based accuracy: 100.0%

2. Phase 6 Pipeline:
   - Bot initialized: ✓
   - Single cycle executed: ✓
   - Simulation mode: ✓ (no real trades)

3. Next steps:
   - Train ML model on the prepared features
   - Save model to 'models/model_latest.joblib'
   - Enable live trading in Phase6Config
   - Run continuous mode for automated trading

✓ Notebook execution complete!
